# MedficientSAM HAM10K Full Evaluation\n\nUse **Runtime > Change runtime type > GPU** before running. This notebook evaluates the uploaded HAM10K image ZIP against the Kaggle lesion masks using tight boxes derived from the ground-truth masks.

In [ ]:
!nvidia-smi\n!python --version

## Clone Repo

In [ ]:
%cd /content\n!rm -rf medificentSAM\n!git clone https://github.com/akashkr077/medificentSAM.git\n%cd /content/medificentSAM

## Install Runtime Dependencies\n\nThis installs only the dependencies needed for `infer_scripts/infer_torch.py` and the HAM10K evaluation helper.

In [ ]:
!pip install -q timm onnx onnxsim onnxruntime lru-dict rootutils hydra-core monai fvcore torchinfo pandas matplotlib pillow tqdm

## Mount Google Drive\n\nPut these files in a Drive folder, for example `MyDrive/medficientsam-data/`:\n\n- `ham10k-20260430T164238Z-3-001.zip`\n- `dataset.zip` from Kaggle lesion segmentations\n- `model.pth` or the repo's `weights/finetuned-l1-augmented/torch/model.pth` file

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

## Set File Paths

In [ ]:
DATA_DIR = '/content/drive/MyDrive/medficientsam-data'\nHAM10K_ZIP = f'{DATA_DIR}/ham10k-20260430T164238Z-3-001.zip'\nMASK_ZIP = f'{DATA_DIR}/dataset.zip'\nMODEL = f'{DATA_DIR}/model.pth'\nOUTPUT_DIR = f'{DATA_DIR}/ham10k_eval_outputs'\n\n!ls -lh "$HAM10K_ZIP" "$MASK_ZIP" "$MODEL"

## Quick GPU Smoke Test\n\nThis runs 20 images first. If it succeeds, run the full cell below.

In [ ]:
!python scripts/ham10k_full_eval.py \\\n  --ham10k-zip "$HAM10K_ZIP" \\\n  --mask-zip "$MASK_ZIP" \\\n  --model "$MODEL" \\\n  --work-dir /content/ham10k_eval_work_smoke \\\n  --output-dir "$OUTPUT_DIR/smoke_20" \\\n  --device cuda \\\n  --limit 20

## Full 5,015 Image Evaluation\n\nExpected free-Colab T4 time: roughly 20-45 minutes for inference, plus setup/conversion time.

In [ ]:
!python scripts/ham10k_full_eval.py \\\n  --ham10k-zip "$HAM10K_ZIP" \\\n  --mask-zip "$MASK_ZIP" \\\n  --model "$MODEL" \\\n  --work-dir /content/ham10k_eval_work_full \\\n  --output-dir "$OUTPUT_DIR/full_5015" \\\n  --device cuda

## View Summary

In [ ]:
import pandas as pd\nsummary = pd.read_csv(f'{OUTPUT_DIR}/full_5015/metrics/summary.csv')\nsummary